# sat_arl — Satélite ARL

Este notebook construye el **satélite ARL**, que consolida la información de personas
registradas en el sistema AS400, unificando dos roles (Tomador y Asegurado)
en una sola tabla sin duplicados.

## Regla de negocio

- **1 registro por TOMADOR por contrato** (persona jurídica o natural contratante).
- **1 registro por ASEGURADO por contrato** (empleados vinculados a la póliza ARL).
- No existe rol BENEFICIARIO en ARL.
- Deduplicación final por `(tipo_documento, numero_documento, contrato, rol)`
  priorizando la fila con `actividad_economica` más completa y `fecha_cargue` más reciente.

## Tablas fuente

| Alias | Tabla AS400 | Qué contiene |
|---|---|---|
| `afaaf` | `EXT_AAAFAAF0` | Datos del Tomador (nombre completo, doc, contacto, dirección) |
| `afnaf` | `EXT_AAAFNAF0` | Contrato (AFNAFI, AFNNIT, TIPO_CONTRATO, ATDP, ACTIVIDAD_ECONOMICA, ESTADO, fechas) |
| `ciuaf` | `EXT_AACIUAF0` | Catálogo de ciudades → departamento |
| `tatadf` | `EXT_AATATDF0` | Catálogo tipo de documento → TIPO_PERSONA (JUR/NAT) |
| `icoaf` | `EXT_AAICOAF0` | Clave asesor por contrato |
| `empaf` | `EXT_AAEMPAF0` | Datos del Asegurado (nombre, doc, contacto, fechas) |

## Tabla destino

`axa_col_slv_dv.stg_cliente.sat_arl`

## Ejecución — Crear el satélite ARL

La siguiente celda crea (o reemplaza) la tabla `axa_col_slv_dv.stg_cliente.sat_arl`.
Se aplica deduplicación por `fecha_cargue` en cada tabla fuente antes de los JOINs,
siguiendo el mismo patrón de las queries de referencia SQL Server.

> ⚠️ **Importante:** Este proceso borra y recrea la tabla completa cada vez que se ejecuta.

In [ ]:
%sql

CREATE OR REPLACE TABLE axa_col_slv_dv.stg_cliente.sat_arl
USING DELTA AS

WITH

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 1 — Deduplicación de tablas fuente (filtro: últimos 3 meses)
-- ══════════════════════════════════════════════════════════════════════════════

afaaf AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY AFAAFI, AFANIT
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.as_arafild0_aaafaaf0
        WHERE fecha_cargue >= ADD_MONTHS(CURRENT_DATE(), -3)
    ) t WHERE rn = 1
),

afnaf AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY AFNAFI
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.as_arafild0_aaafnaf0
    ) t WHERE rn = 1
),

ciuaf AS (
    SELECT DISTINCT CIUCCI, CIUCDP
    FROM axa_col_slv_dv.core_as400.as_arafild0_aaciuaf0
),

icoaf AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY ICOAFI
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.as_arafild0_aaicoaf0
    ) t WHERE rn = 1
),

empaf AS (
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY EMPAFI, EMPIDE
                   ORDER BY fecha_cargue DESC
               ) AS rn
        FROM axa_col_slv_dv.core_as400.as_arafild0_aaempaf0
        WHERE EMPEST = 0
          AND fecha_cargue >= ADD_MONTHS(CURRENT_DATE(), -3)
    ) t WHERE rn = 1
),

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 2 — Rol: TOMADOR
-- celular   = AFATE1 (teléfono principal)
-- celular_2 = AFATE2 (teléfono secundario; se expande en BLOQUE 5)
-- ══════════════════════════════════════════════════════════════════════════════

tomador AS (
    SELECT
        A.AFATDO                                                           AS tipo_documento,
        CAST(A.AFANIT AS STRING)                                           AS numero_documento,
        CAST(NULL AS STRING)                                               AS primer_nombre,
        CAST(NULL AS STRING)                                               AS segundo_nombre,
        CAST(NULL AS STRING)                                               AS primer_apellido,
        CAST(NULL AS STRING)                                               AS segundo_apellido,
        A.AFARSO                                                           AS nombre_completo_razon_social,
        A.AFAEML                                                           AS email,
        NULLIF(TRIM(CAST(A.AFATE1 AS STRING)), '')                        AS celular,
        NULLIF(TRIM(CAST(A.AFATE2 AS STRING)), '')                        AS celular_2,
        A.AFADIR                                                           AS direccion_residencial,
        A.AFACIU                                                           AS ciudad_residencia,
        C.CIUCDP                                                           AS departamento,
        'COL'                                                              AS pais,
        CAST(NULL AS DATE)                                                 AS fecha_nacimiento,
        CAST(NULL AS STRING)                                               AS genero,
        CAST(NULL AS STRING)                                               AS estado_civil,
        B.AFNFU3                                                           AS ATDP,
        B.AFNACT                                                           AS actividad_economica,
        CAST(B.AFNAFI AS STRING)                                           AS contrato,
        B.AFNTAF                                                           AS tipo_contrato,
        F.ICOCLA                                                           AS clave,
        CAST(B.AFNEST AS STRING)                                           AS estado,
        TRY_TO_DATE(CAST(B.AFNFIV AS STRING), 'yyyyMMdd')                 AS fecha_inicio_vigencia,
        TRY_TO_DATE(CAST(B.AFNRET AS STRING), 'yyyyMMdd')                 AS fecha_retiro,
        TRY_TO_DATE(CAST(B.AFNREV AS STRING), 'yyyyMMdd')                 AS fecha_revinculacion,
        CAST(CASE WHEN A.AFATDO = 2 THEN 'JUR' ELSE 'NAT' END AS STRING) AS tipo_persona,
        A.fecha_cargue                                                     AS FEC_ACTUALIZACION,
        'TOMADOR'                                                          AS rol
    FROM afaaf A
    LEFT JOIN afnaf B ON CONCAT(CAST(A.AFAAFI AS STRING), CAST(A.AFANIT AS STRING))
                       = CONCAT(CAST(B.AFNAFI AS STRING), CAST(B.AFNNIT AS STRING))
    LEFT JOIN ciuaf C ON A.AFACIU = C.CIUCCI
    LEFT JOIN icoaf F ON B.AFNAFI = F.ICOAFI
),

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 3 — Rol: ASEGURADO
-- celular = EMPTEL (único teléfono disponible para asegurados)
-- ══════════════════════════════════════════════════════════════════════════════

asegurado AS (
    SELECT
        A.EMPTDO                                                           AS tipo_documento,
        CAST(A.EMPIDE AS STRING)                                           AS numero_documento,
        A.EMPNO1                                                           AS primer_nombre,
        A.EMPNO2                                                           AS segundo_nombre,
        A.EMPAP1                                                           AS primer_apellido,
        A.EMPAP2                                                           AS segundo_apellido,
        TRIM(CONCAT_WS(' ',
            NULLIF(TRIM(A.EMPNO1), ''),
            NULLIF(TRIM(A.EMPNO2), ''),
            NULLIF(TRIM(A.EMPAP1), ''),
            NULLIF(TRIM(A.EMPAP2), '')
        ))                                                                 AS nombre_completo_razon_social,
        A.EMPEML                                                           AS email,
        NULLIF(TRIM(CAST(A.EMPTEL AS STRING)), '')                        AS celular,
        CAST(NULL AS STRING)                                               AS celular_2,
        A.EMPDIR                                                           AS direccion_residencial,
        A.EMPCIU                                                           AS ciudad_residencia,
        C.CIUCDP                                                           AS departamento,
        'COL'                                                              AS pais,
        TRY_TO_DATE(CAST(A.EMPFNA AS STRING), 'yyyyMMdd')                 AS fecha_nacimiento,
        A.EMPSEX                                                           AS genero,
        A.EMPESC                                                           AS estado_civil,
        A.EMPEPA                                                           AS ATDP,
        '0010'                                                             AS actividad_economica,
        CAST(B.AFNAFI AS STRING)                                           AS contrato,
        A.EMPTVE                                                           AS tipo_contrato,
        F.ICOCLA                                                           AS clave,
        CAST(A.EMPEST AS STRING)                                           AS estado,
        TRY_TO_DATE(CAST(A.EMPING AS STRING), 'yyyyMMdd')                 AS fecha_inicio_vigencia,
        TRY_TO_DATE(CAST(A.EMPRET AS STRING), 'yyyyMMdd')                 AS fecha_retiro,
        TRY_TO_DATE(CAST(A.EMPREV AS STRING), 'yyyyMMdd')                 AS fecha_revinculacion,
        'NAT'                                                              AS tipo_persona,
        A.fecha_cargue                                                     AS FEC_ACTUALIZACION,
        'ASEGURADO'                                                        AS rol
    FROM empaf A
    LEFT JOIN afnaf B ON A.EMPAFI = B.AFNAFI
    LEFT JOIN ciuaf C ON A.EMPCIU = C.CIUCCI
    LEFT JOIN icoaf F ON B.AFNAFI = F.ICOAFI
),

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 4 — Dedup por (tipo_documento, numero_documento, contrato, rol)
-- Se deduplicar ANTES de expandir celular_2 para evitar filas fantasma.
-- ══════════════════════════════════════════════════════════════════════════════

all_roles AS (
    SELECT * FROM tomador
    UNION ALL
    SELECT * FROM asegurado
),

dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY tipo_documento, numero_documento, contrato, rol
                   ORDER BY
                       CASE WHEN actividad_economica IS NULL
                                 OR TRIM(actividad_economica) = '' THEN 1 ELSE 0 END,
                       FEC_ACTUALIZACION DESC
               ) AS _rn
        FROM all_roles
    ) t
    WHERE _rn = 1
)

-- ══════════════════════════════════════════════════════════════════════════════
-- BLOQUE 5 — Expansión de celular_2
-- Fila 1 : celular principal (siempre)
-- Fila 2 : celular_2 solo si existe Y es diferente al celular principal
--          → evita duplicados cuando AFATE1 = AFATE2
-- ══════════════════════════════════════════════════════════════════════════════

SELECT
    tipo_documento,
    numero_documento,
    primer_nombre,
    segundo_nombre,
    primer_apellido,
    segundo_apellido,
    nombre_completo_razon_social,
    email,
    celular,
    direccion_residencial,
    ciudad_residencia,
    departamento,
    pais,
    fecha_nacimiento,
    genero,
    estado_civil,
    ATDP,
    actividad_economica,
    contrato,
    tipo_contrato,
    clave,
    estado,
    fecha_inicio_vigencia,
    fecha_retiro,
    fecha_revinculacion,
    tipo_persona,
    FEC_ACTUALIZACION,
    rol,
    CONCAT(CAST(tipo_documento AS STRING), CAST(numero_documento AS STRING)) AS llave_negocio
FROM dedup

UNION ALL

SELECT
    tipo_documento,
    numero_documento,
    primer_nombre,
    segundo_nombre,
    primer_apellido,
    segundo_apellido,
    nombre_completo_razon_social,
    email,
    celular_2                                                              AS celular,
    direccion_residencial,
    ciudad_residencia,
    departamento,
    pais,
    fecha_nacimiento,
    genero,
    estado_civil,
    ATDP,
    actividad_economica,
    contrato,
    tipo_contrato,
    clave,
    estado,
    fecha_inicio_vigencia,
    fecha_retiro,
    fecha_revinculacion,
    tipo_persona,
    FEC_ACTUALIZACION,
    rol,
    CONCAT(CAST(tipo_documento AS STRING), CAST(numero_documento AS STRING)) AS llave_negocio
FROM dedup
WHERE celular_2 IS NOT NULL
  AND TRIM(celular_2) <> ''
  AND TRIM(celular_2) <> TRIM(COALESCE(celular, ''))

## Validación — Verificar el resultado

In [ ]:
%sql
-- Total de filas cargadas por rol
SELECT
    rol,
    COUNT(*)                          AS total_filas,
    COUNT(DISTINCT numero_documento)  AS personas_unicas,
    COUNT(DISTINCT contrato)          AS contratos_unicos
FROM axa_col_slv_dv.stg_cliente.sat_arl
GROUP BY rol
ORDER BY rol

In [ ]:
%sql
-- Verificar que no haya duplicados por (tipo_doc, num_doc, contrato, rol)
SELECT
    tipo_documento,
    numero_documento,
    contrato,
    rol,
    COUNT(*) AS veces
FROM axa_col_slv_dv.stg_cliente.sat_arl
GROUP BY tipo_documento, numero_documento, contrato, rol
HAVING COUNT(*) > 1
ORDER BY veces DESC
LIMIT 10

In [ ]:
%sql
-- Vista previa de los primeros 5 registros
SELECT
    rol,
    tipo_documento,
    numero_documento,
    primer_nombre,
    primer_apellido,
    nombre_completo_razon_social,
    correo,
    telefono_1,
    telefono_2,
    ciudad_residencia,
    departamento,
    pais,
    actividad_economica,
    contrato,
    tipo_contrato,
    clave,
    estado,
    fecha_inicio,
    fecha_retiro,
    fecha_revinculacion,
    tipo_persona,
    FEC_ACTUALIZACION
FROM axa_col_slv_dv.stg_cliente.sat_arl
LIMIT 5